In [7]:
import os
from pathlib import Path
import sys


def find_project_root(target_dir="configs"):
    current_path = Path.cwd().resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / target_dir).exists():
            return parent
    return current_path


ROOT = find_project_root()
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# print(f"Project root set to: {ROOT}")

In [8]:
import json
from nltk.stem import WordNetLemmatizer
import torch
import yaml

from src.pathfind.morph_a_star import morph_a_star_generator
from src.utils.load import load_model_for_inference
from src.visualize.render import render

In [9]:
CONFIG_PATH = ROOT / "configs" / "config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)


embedding_metadata_path = (
    Path(cfg["data"]["dir"]) / cfg["data"]["embedding_metadata_path"]
)
centroids_path = Path(cfg["data"]["dir"]) / cfg["data"]["centroids_path"]
centroid_metadata_path = (
    Path(cfg["data"]["dir"]) / cfg["data"]["centroid_metadata_path"]
)
checkpoint_for_inference_path = (
    Path(cfg["checkpoints"]["dir"])
    / cfg["checkpoints"]["checkpoint_for_inference_path"]
)
semantic_manifold_path = (
    Path(cfg["data"]["dir"]) / cfg["data"]["semantic_manifold_path"]
)

In [10]:
with open(centroid_metadata_path, "r") as f:
    centroid_metadata = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = load_model_for_inference(
    checkpoint_path=checkpoint_for_inference_path, cfg=cfg, device=device
)
lemmatizer = WordNetLemmatizer()

k_neighbors = cfg["pathfind"]["kdtree"]["k_neighbors"]
max_neighbor_distance = cfg["pathfind"]["kdtree"]["max_neighbor_distance"]
k_cutoff = cfg["pathfind"]["kdtree"]["k_cutoff"]
temperature = cfg["pathfind"]["kdtree"]["temperature"]
max_expansions = cfg["pathfind"]["astar"]["max_expansions"]
weights = cfg["pathfind"]["astar"]["weights"]
allow_immediate_reach = cfg["pathfind"]["astar"]["allow_immediate_reach"]

Using device: cuda


In [11]:
# pairs = [
#     # Association
#     ("king", "queen"),
#     ("sunny", "windy"),
#     ("bread", "coffee"),
#     ("animal", "human"),
#     ("person", "company"),
#     # Contrast
#     ("bright", "dark"),
#     ("cold", "hot"),
#     ("happy", "sad"),
#     ("strong", "weak"),
#     ("lose", "win"),
#     ("war", "peace"),
#     ("ice", "lava"),
#     ("mayor", "slave"),
#     # Geographics
#     ("tunnel", "sea"),
#     ("forest", "desert"),
#     ("city", "village"),
#     ("america", "asia"),
#     # Transformation
#     ("ant", "elephant"),
#     ("leaf", "jungle"),
#     ("disturbance", "catastrophe"),
#     ("tree", "paper"),
#     ("imagine", "possess"),
#     ("birth", "death"),
#     # Abstract
#     ("wave", "atom"),
#     ("freedom", "prison"),
#     ("justice", "scale"),
#     ("truth", "illusion"),
#     ("romance", "power"),
#     ("tale", "nightmare"),
#     ("sense", "flow"),
#     # Weak association
#     ("tea", "universe"),
#     ("history", "nose"),
#     ("drink", "verb"),
#     ("python", "pond"),
#     # Polysemy
#     ("chip", "computer"),
#     ("chip", "chocolate"),
#     ("window", "surf"),
#     ("grass", "surf"),
# ]

In [12]:
pairs = [("grass", "surf")]

In [13]:
for i, (word_start, word_end) in enumerate(pairs):
    gen = morph_a_star_generator(
        word_start=word_start,
        word_end=word_end,
        model=model,
        lemmatizer=lemmatizer,
        k_neighbors=k_neighbors,
        max_neighbor_distance=max_neighbor_distance,
        k_cutoff=k_cutoff,
        temperature=0,  # deterministic
        weights=weights,
        max_expansions=max_expansions,
        allow_immediate_reach=allow_immediate_reach,
        centroids_path=centroids_path,
        centroid_metadata=centroid_metadata,
        device=device,
    )
    result = next(gen)

    if result["status"] == 2:
        word_sense_path = result["word_sense_path"]
        word_path = [x.split("_")[0] for x in word_sense_path]  # type: ignore
        centroid_path = result["centroid_path"]
        dynamic_path = [
            x.split("_")[0] if word_path.count(x.split("_")[0]) == 1 else x
            for x in word_sense_path  # type: ignore
        ]
        print(" → ".join(dynamic_path))
        if i == len(pairs) - 1:
            fig = render(
                word_path,
                centroid_path,
                semantic_manifold_path,
                centroid_metadata,
                show_info=True,
            )
            fig.show()

grass → grassy → sandy → beach → surf
